# 04 — Baseline

Usa o `build_pipeline()` de `src/features/preparation.py` (PR #11) — o pré-processamento
(imputação, scaler, encoder, engenharia estrutural) roda dentro do próprio `Pipeline`,
então o `fit` de cada fold da validação cruzada só vê o treino daquele fold, sem vazar
para o teste.

## 1. Setup

In [1]:
import sys
from pathlib import Path

import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

# E402: o sys.path.insert acima precisa rodar antes destes imports,
# caso contrário o notebook não encontra o pacote src ao rodar de dentro de notebooks/.
from src.config import (  # noqa: E402
    MLFLOW_EXPERIMENT_NAME,
    MODELS_DIR,
    PROCESSED_DATA_DIR,
    configurar_mlflow_tracking,
    iniciar_run,
    limpar_runs_anteriores,
)
from src.features.preparation import build_pipeline, filtrar_censura, separar_alvo  # noqa: E402

RANDOM_STATE = 42

## 1b. Métricas (F1 / AUC-ROC / PR-AUC)

Definidas aqui, direto no notebook — sem acurácia isolada, que engana com 26,5% de
churn (um classificador "sempre não" já acerta 73,5%).

In [2]:
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score


def calcular_metricas(y_true, y_pred, y_proba) -> dict[str, float]:
    return {
        "f1": f1_score(y_true, y_pred),
        "auc_roc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
    }


def formatar_metricas(nome_modelo: str, metricas: dict[str, float]) -> str:
    return (
        f"{nome_modelo}: "
        f"F1={metricas['f1']:.3f}  "
        f"AUC-ROC={metricas['auc_roc']:.3f}  "
        f"PR-AUC={metricas['pr_auc']:.3f}"
    )

## 2. Carregar a base já processada

In [3]:
df = pd.read_parquet(PROCESSED_DATA_DIR / "telco_churn_processed.parquet")
print("Shape bruto:", df.shape)

Shape bruto: (7043, 61)


## 3. Filtrar censura e separar alvo

`filtrar_censura(remover_joined=False)` mantém os clientes "Joined" — decisão validada
em `src/features/preparation.py`: removê-los cria uma separação artificial otimista no
ROC-AUC (testado, não é uma escolha arbitrária deste notebook).

In [4]:
df_modelagem, df_censurados = filtrar_censura(df, remover_joined=False)
X, y = separar_alvo(df_modelagem)

print("X shape:", X.shape)
print("y distribuição:", y.value_counts().to_dict())
print(f"Taxa de churn: {y.mean():.1%}")

X shape: (7043, 60)
y distribuição: {0: 5174, 1: 1869}
Taxa de churn: 26.5%


## 4. Split treino/teste

`stratify=y` mantém a mesma proporção de churn (~26,5%) no treino e no teste — sem isso,
o sorteio da divisão pode distorcer essa proporção e enviesar a métrica final.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("X_train:", X_train.shape, " X_test:", X_test.shape)
print("Proporção de churn — treino:", y_train.mean().round(3), " teste:", y_test.mean().round(3))

X_train: (5634, 60)  X_test: (1409, 60)
Proporção de churn — treino: 0.265  teste: 0.265


## 5. Validação cruzada (só no treino)

`class_weight="balanced"` compensa o desbalanceamento (73,5%/26,5%) sem precisar de
reamostragem. `StratifiedKFold` mantém a proporção de churn em cada um dos 5 folds.

O `pipeline` completo (pré-processamento + modelo) é o que entra no `cross_validate` —
assim o `fit` do scaler/encoder roda dentro de cada fold, nunca vazando informação do
fold de validação para o de treino. Usamos `cross_validate` com um dicionário de
`scoring` (em vez de chamar `cross_val_score` três vezes) para não repetir o `fit` do
pipeline 15x à toa (5 folds × 3 métricas) — com `cross_validate`, cada fold treina uma
única vez e calcula as três métricas sobre esse mesmo treino.

In [6]:
modelo = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)
pipeline = build_pipeline(modelo=modelo)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {"f1": "f1", "auc_roc": "roc_auc", "pr_auc": "average_precision"}
resultados_cv = cross_validate(pipeline, X_train, y_train, cv=skf, scoring=scoring)

scores_f1 = resultados_cv["test_f1"]
scores_auc = resultados_cv["test_auc_roc"]
scores_prauc = resultados_cv["test_pr_auc"]

print(f"F1:      {scores_f1.mean():.3f} (+/- {scores_f1.std():.3f})")
print(f"AUC-ROC: {scores_auc.mean():.3f} (+/- {scores_auc.std():.3f})")
print(f"PR-AUC:  {scores_prauc.mean():.3f} (+/- {scores_prauc.std():.3f})")

F1:      0.679 (+/- 0.016)
AUC-ROC: 0.896 (+/- 0.007)
PR-AUC:  0.757 (+/- 0.013)


## 6. Treino final e avaliação única no teste

O teste só é usado agora, uma única vez — nunca foi visto durante a validação cruzada
acima. Se essas métricas destoarem muito das da validação cruzada, é sinal de alerta
(overfitting ou vazamento); próximas é o esperado.

In [7]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

metricas_teste = calcular_metricas(y_test, y_pred, y_proba)
print(formatar_metricas("Baseline (Regressão Logística) — teste", metricas_teste))

Baseline (Regressão Logística) — teste: F1=0.694  AUC-ROC=0.908  PR-AUC=0.766


## 7. Comparação com o "baseline preguiçoso"

Com 26,5% de churn, um modelo que sempre prevê "não-churn" já acerta 73,5% de acurácia
sem aprender nada. Por isso a avaliação principal usa F1/AUC-ROC/PR-AUC (seção 5 e 6),
não acurácia isolada — esse número abaixo é só para dar contexto de quanto o modelo
está de fato acima do chute ingênuo.

In [8]:
from sklearn.metrics import accuracy_score

acuracia_naive = 1 - y_test.mean()
acuracia_modelo = accuracy_score(y_test, y_pred)

print(f"Acurácia do 'sempre não-churn': {acuracia_naive:.3f}")
print(f"Acurácia do modelo:             {acuracia_modelo:.3f}")

Acurácia do 'sempre não-churn': 0.735
Acurácia do modelo:             0.796


## 8. Registrar o modelo no MLflow (com fallback local)

O `mlflow.sklearn.log_model()` já salva o `pipeline` inteiro (pré-processamento +
modelo) como artifact da run — com signature, exemplo de input e o
`requirements.txt`/`conda.yaml` com as versões exatas das libs usadas. Isso registra
tanto os parâmetros/métricas quanto o modelo em si numa única run, no MLflow local ou
no DagsHub (dependendo do `.env` — ninguém precisa ter o DagsHub configurado para
rodar este notebook, cai automaticamente para o modo local via SQLite `mlflow.db`).

Como o registro no MLflow já cobre o salvamento do modelo, `models/*.joblib` só é
gravado como **fallback** — se o bloco do MLflow falhar por qualquer motivo (sem
`mlflow`/`dagshub` instalado, erro de rede etc.), o `except` salva localmente via
`joblib.dump()` para não perder o modelo treinado.

In [9]:
NOME_MODELO = "baseline_logistic_regression"

try:
    import mlflow
    import mlflow.sklearn

    configurar_mlflow_tracking()
    mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
    limpar_runs_anteriores([NOME_MODELO])
    with iniciar_run("notebooks/04_baseline.ipynb", run_name=NOME_MODELO):
        mlflow.log_params(
            {
                "modelo": "LogisticRegression",
                "class_weight": "balanced",
                "random_state": RANDOM_STATE,
            }
        )
        mlflow.log_metrics(
            {
                "cv_f1": scores_f1.mean(),
                "cv_f1_std": scores_f1.std(),
                "cv_auc_roc": scores_auc.mean(),
                "cv_auc_roc_std": scores_auc.std(),
                "cv_pr_auc": scores_prauc.mean(),
                "cv_pr_auc_std": scores_prauc.std(),
            }
        )
        mlflow.log_metrics({f"teste_{k}": v for k, v in metricas_teste.items()})
        mlflow.sklearn.log_model(pipeline, name="modelo", serialization_format="cloudpickle")
    print("Run registrada no MLflow, com o modelo salvo como artifact.")
except Exception as erro:
    print("Não foi possível registrar no MLflow:", erro)
    print("Salvando o modelo localmente como fallback...")
    try:
        MODELS_DIR.mkdir(parents=True, exist_ok=True)
        caminho_modelo = Path(MODELS_DIR) / f"{NOME_MODELO}.joblib"
        joblib.dump(pipeline, caminho_modelo)
        print("Modelo salvo em:", caminho_modelo)
    except Exception as erro_local:
        print("Também não foi possível salvar o modelo localmente:", erro_local)

Accessing as ThiagoZulian

Initialized MLflow to track repo "ThiagoZulian/Grupo-57-Machine-Learning-Engineering"

Repository ThiagoZulian/Grupo-57-Machine-Learning-Engineering initialized!

2026/08/19 12:19:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/08/19 12:19:54 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run baseline_logistic_regression at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0/runs/902066c8f13a486895e136366cad5d23
🧪 View experiment at: https://dagshub.com/ThiagoZulian/Grupo-57-Machine-Learning-Engineering.mlflow/#/experiments/0


Run registrada no MLflow, com o modelo salvo como artifact.


Nota: a célula acima ganhou um passo de idempotência e a fixação explícita de `mlflow.source.name` depois da execução original deste notebook (correção aplicada em resposta a runs duplicadas encontradas no MLflow - ver `docs/decisions.md`, ADR-004). O resultado numérico do baseline não muda; uma reexecução limpa deve reproduzir as mesmas métricas já documentadas na seção 9.

## 9. Conclusão

| Métrica | Validação cruzada (5 folds) | Teste |
|---|---|---|
| F1 | ~0.679 | ~0.694 |
| AUC-ROC | ~0.896 | ~0.908 |
| PR-AUC | ~0.757 | ~0.766 |

*(valores de referência da última execução — os números exatos ao rodar este notebook
devem ficar próximos, com pequena variação natural.)*

Este é o modelo final do projeto (ver ADR-004 em docs/decisions.md).